# Middleware
Middleware 提供了hook函数来实现控制agent的行为的方式：
- 日志分析
- 重试，回退等逻辑
- 限制速率，检测等

一些内置的中间件

| Middleware中间件     | Description描述                                              |
| :------------------- | :----------------------------------------------------------- |
| Summarization        | Automatically summarize conversation history when approaching token limits.接近代币限制时自动总结对话记录。 |
| Human-in-the-loop    | Pause execution for human approval of tool calls.暂停执行以供人工批准工具调用。 |
| Model call limit     | Limit the number of model calls to prevent excessive costs.限制模型调用次数，以防止过高成本。 |
| Tool call limit      | rol tool execution by limiting call counts.通过限制呼叫次数来控制工具执行。 |
| Model fallback       | Automatically fallback to alternative models when primary fails.当主模式失败时，会自动回退到其他模式。 |
| PII detection        | Detect and handle Personally Identifiable Information (PII).检测并处理个人身份信息（PII）。 |
| To-do list           | Equip agents with task planning and tracking capabilities.为客服人员配备任务规划和跟踪能力。 |
| LLM tool selector    | Use an LLM to select relevant tools before calling main model.在调用主模型之前，先用LLM选择相关工具。 |
| Tool retry           | Automatically retry failed tool calls with exponential backoff.用指数回撤自动重试失败的工具调用。 |
| Model retry          | Automatically retry failed model calls with exponential backoff.自动用指数退回方式重试失败的模型调用。 |
| LLM tool emulator    | Emulate tool execution using an LLM for testing purposes.用LLM模拟工具执行以进行测试。 |
| Context editing      | Manage conversation context by trimming or clearing tool uses.通过修剪或清理工具使用来管理对话上下文。 |
| Provider tool search | Defer tools behind providers’ server-side tool search, surfacing them on demand.将工具推迟到服务提供商的服务器端工具搜索后，按需展示。 |
| Shell tool           | Expose a persistent shell session to agents for command execution.向代理开放一个持久的壳会话以执行命令。 |
| File search          | Provide Glob and Grep search tools over filesystem files.在文件系统文件上提供 Glob 和 Grep 搜索工具。 |
| Filesystem           | Provide agents with a filesystem for storing context and long-term memories.为代理提供存储上下文和长期记忆的文件系统。 |
| Subagent             | Add the ability to spawn subagents.增加生成子代理人的能力。  |

In [1]:
# Summarization MiddleWare
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware  import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY"),
)
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm, # 用于生成摘要的模型（通常选便宜、快的模型）
            trigger=("messages",4),  # 触发条件：对话累计达到 10 条消息时，开始压缩历史
            keep=("messages",2)   # 保留最近 4 条消息不被压缩，其余的会被总结成摘要
        )
    ]
)

In [18]:
### Run with thread id
config = {"configurable":{"thread_id":"test-111"}}

questions = [
    "What is 2+2 ?",
    "What is 2*2 ?",
    "What is 2-2 ?",
    "What is 15-2 ?",
    "What is 100/5 ?"
]

for question in questions:
    response = agent.invoke({"messages":[HumanMessage(content=question)]},config)
    print(f"Message:{response}")
    print(f"Message:{len(response['messages'])}")

Message:{'messages': [HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}, id='c62cc6d3-72d9-4527-be73-a1eeda82e94e'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1b84-c304-7383-a0c0-06638c6c32c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Message:2
Message:{'messages': [HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}, id='c62cc6d3-72d9-4527-be73-a1eeda82e94e'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1b84-c304-7383-a0c0-06638c6c32c0-0', tool_calls=[], invalid_tool_calls=[], usage_

In [23]:
# token size
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""
    Hotels in {city}:
    1.Grand Hotel - 5 star,$350/night, spa,pool, gym
    2. City Inn - 4 star, $180/night, business center
    3.Budget Stay - 3 star, $75/night, free wifi
    """
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        # 压缩上下文
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-123"}}

# token 计数器
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4个字符约等于1token

In [24]:
cities = ["巴黎","伦敦","东京","纽约","大阪","悉尼"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"寻找旅馆在 {city}")]},
        config = config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~ {tokens} tokens , {len(response['messages'])} messages")   
    print(f"{(response['messages'])}")



巴黎: ~ 78 tokens , 4 messages
[HumanMessage(content='寻找旅馆在 巴黎', additional_kwargs={}, response_metadata={}, id='a7a8d0a7-6193-4ba6-b74f-cc92b88f700f'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "\\u5df4\\u9ece"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1c28-d8e4-7691-9701-85b54cb38cc6-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': '巴黎'}, 'id': '137c92b1-3e52-4a15-bde2-93e955d351da', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 15, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='\n    Hotels in 巴黎:\n    1.Grand Hotel - 5 star,$350/night, spa,pool, gym\n    2. City Inn - 4 star, $180/night, business center\n    3.Budget Stay - 3 star, $75/night, free wifi\n    ', name='search_hotels', id='31a1c977-e31f-4549

In [6]:
# fraction
import fractions
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""
    Hotels in {city}:
    1.Grand Hotel - 5 star,$350/night, spa,pool, gym
    2. City Inn - 4 star, $180/night, business center
    3.Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        # 压缩上下文
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction",0.0003), #  当消息 token 数达到模型 max_input_tokens 的 0.03% 时,触发摘要压缩
            keep=("fraction",0.0002) #  压缩后保留相当于 max_input_tokens 0.02% 的最近上下文
        )
    ]
)

config = {"configurable":{"thread_id":"test-789789"}}

# token 计数器
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars //4 # 4个字符约等于1token

# 1048576 
cities = ["巴黎","伦敦","东京","纽约","大阪","悉尼"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"寻找旅馆在 {city}")]},
        config = config
    )
    
    tokens = count_tokens(response["messages"])
    fraction = tokens / 1048576
    print(f"{city}: ~ {tokens} tokens ({fraction:.4%}) , {len(response['messages'])} messages")   
    print(f"{(response['messages'])}")


巴黎: ~ 78 tokens (0.0074%) , 4 messages
[HumanMessage(content='寻找旅馆在 巴黎', additional_kwargs={}, response_metadata={}, id='74b2ebdf-1919-49c7-9add-907ad770993c'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "\\u5df4\\u9ece"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1c9d-a50d-78b3-845e-405ae8fb2f1b-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': '巴黎'}, 'id': '66b23012-fbd9-4f38-bf54-d4342107740c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 15, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='\n    Hotels in 巴黎:\n    1.Grand Hotel - 5 star,$350/night, spa,pool, gym\n    2. City Inn - 4 star, $180/night, business center\n    3.Budget Stay - 3 star, $75/night, free wifi\n    ', name='search_hotels', id='d7c8b692

In [38]:
# 人工干预 Human-in-the-loop
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID"""
    print('读邮件')
    return f"Email content for ID:{email_id}"


def send_email_tool(recipient: str , subject: str, body: str) -> str:
    """Mock function to send an email"""
    print('发送邮件')
    return f"email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=llm,
    tools=[send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

config = {"configurable":{"thread_id":"approve-724524"}}





In [39]:
# 发邮件
result = agent.invoke(
    {"messages":[HumanMessage(content="semd email to 123@gmail.com with subject 'hello' and body 'what does the fox say' ")]},
    config = config
)
result

{'messages': [HumanMessage(content="semd email to 123@gmail.com with subject 'hello' and body 'what does the fox say' ", additional_kwargs={}, response_metadata={}, id='45e9ba95-2943-4300-8d28-76be10919379'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "what does the fox say", "subject": "hello", "recipient": "123@gmail.com"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1d04-4611-7092-8a6a-97bd1d533fea-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'what does the fox say', 'subject': 'hello', 'recipient': '123@gmail.com'}, 'id': 'ae2f4ccd-ebc7-44e4-a15f-3c7bc699df26', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 37, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}})],
 '__interrupt__': [Interrupt(value={'action_requests': [{'n

In [ ]:
# 审批通过
from langgraph.types import Command

if "__interrupt__" in result: 
    print("请审批")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config= config
    )
    print(f"结果：{result['messages'][-1].content}")



请审批
发送邮件
结果：I have sent the email to 123@gmail.com with the subject 'hello' and body 'what does the fox say'.


In [16]:
result

{'messages': [HumanMessage(content="semd email to 123@gmail.com with subject 'hello' and body 'what does the fox say' ", additional_kwargs={}, response_metadata={}, id='8a60a51e-0daa-42c2-986b-c89c2308173f'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "what does the fox say", "subject": "hello", "recipient": "123@gmail.com"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1cfc-aae1-71a3-9bda-2a7a4d9791cb-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'what does the fox say', 'subject': 'hello', 'recipient': '123@gmail.com'}, 'id': '5eaf3fb9-d3f2-4fdc-8228-927ae68566e0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 37, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="email sent to 123@gmail.com with subj

In [ ]:
# 审批拒绝
from langgraph.types import Command

if "__interrupt__" in result: 
    print("请审批")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config= config
    )
    print(f"结果：{result['messages'][-1].content}")
print(result)

请审批
结果：I noticed that you asked me to send an email, but you rejected the tool call. Please let me know if you'd like me to try sending the email again.
{'messages': [HumanMessage(content="semd email to 123@gmail.com with subject 'hello' and body 'what does the fox say' ", additional_kwargs={}, response_metadata={}, id='5fd6c0a8-5973-4c3c-85ea-d42090b31d00'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "123@gmail.com", "subject": "hello", "body": "what does the fox say"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1cfe-f473-7a91-a52b-7ae906050ed3-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': '123@gmail.com', 'subject': 'hello', 'body': 'what does the fox say'}, 'id': '668834d0-875b-4144-895e-2ac487511f69', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 

In [40]:
# 编辑操作
from langgraph.types import Command

if "__interrupt__" in result: 
    print("请审批")
    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                     {
                        "type":"edit",
                        "edited_action":{
                            "name":"send_email_tool", # 调用的工具
                            "args":{
                                "recipient":"789@qq.com",
                                "subject":"new Subject",
                                "body":"i see u"
                            }
                        }
                    }
                ]
            }
        ),
        config= config
    )
    print(f"结果：{result['messages'][-1].content}")
print(result)

请审批
发送邮件
结果：I noticed that you asked me to send an email with the subject 'hello' and body 'what does the fox say' to 123@gmail.com. However, I noticed that in the previous turn, I sent an email to 789@qq.com with the subject 'new Subject' and body 'i see u'. Please clarify if you'd like me to send a new email or if you have any other requests.
{'messages': [HumanMessage(content="semd email to 123@gmail.com with subject 'hello' and body 'what does the fox say' ", additional_kwargs={}, response_metadata={}, id='45e9ba95-2943-4300-8d28-76be10919379'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "what does the fox say", "subject": "hello", "recipient": "123@gmail.com"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f1d04-4611-7092-8a6a-97bd1d533fea-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'a